

<center><img alt="" src="images/UIII-Python-WHILE-Header-800x125.png" /></center>

<center><strong><font color="blue"><h2>© Taufik Sutanto</h2></font></strong></center>

## Learning objectives and preparation

By the end of this notebook, you should be able to:

- Explain the initialization, condition, body, and state update of a `while` loop.
- Select an appropriate loop and reason about its termination.
- Use `break`, `continue`, and `while ... else` accurately.
- Apply condition-controlled iteration to accumulation, validation, search, and numerical approximation.
- Test boundary cases and distinguish successful completion from an imposed iteration limit.

We assume familiarity with variables, comparisons, Boolean expressions, `if` statements, and basic `for` loops. Later examples use simple functions and exception handling, with explanations provided where needed. All examples use Python 3 and its standard library. Run the cells in order; later cells reuse earlier definitions.

### Review questions

1. ...

## 1. Three useful forms of repetition

| Form | When is continuation decided? | Typical Python expression |
|---|---|---|
| Iterable-controlled iteration | As items are obtained from an iterable | `for item in values:` |
| Pre-test, condition-controlled iteration | Before every execution of the body | `while condition:` |
| Post-test iteration | After the body has executed at least once | `while True:` followed by a conditional `break` |

A `for` loop is particularly useful when we want to process a sequence or a fixed range. Its number of iterations need not always be known in advance: an iterator may produce values dynamically. A `while` loop expresses repetition governed directly by a condition. Although it can also perform a fixed number of repetitions, it is especially useful when the required number emerges during execution.

Python has no separate `do ... while` or `repeat ... until` statement. We will implement post-test behavior explicitly later.

## 2. The structure of a `while` loop

```python
state = initial_value
while continuation_condition:
    # Perform the required work.
    # Update the state that affects the condition.
```

The condition is evaluated **before** the body. If it is true, Python executes the body and returns to the condition. If it is false, execution proceeds after the loop. Consequently, the body may execute zero times.

The original notes refer to a “stopper.” To avoid ambiguity, we distinguish a **continuation condition**, which must remain true for the loop to run, from a **stopping condition**, which indicates that the work should stop. In an ordinary `while` loop, normal termination occurs when the continuation condition becomes **false**. A `break` statement can also terminate the loop.

| Component | Question to ask |
|---|---|
| Initialization | What must be defined before the first condition check? |
| Condition | Under what circumstances should another iteration occur? |
| Body | What work is repeated? |
| State update | What changes so that the loop can make progress? |

A state update may be an arithmetic operation, a new input, or a change in an external resource. It need not always be a counter increment.

In [1]:
remaining = 7
while remaining > 0:
    print(remaining, end=", ")
    remaining -= 1
print("Done.")
print("Final state:", remaining)

7, 6, 5, 4, 3, 2, 1, Done.
Final state: 0


### Trace the countdown

| Condition check | `remaining` before the body | Output | `remaining` after the body |
|---|---:|---:|---:|
| First | 7 | 7 | 6 |
| Second | 6 | 6 | 5 |
| … | … | … | … |
| Seventh | 1 | 1 | 0 |
| Eighth | 0 | No body execution | 0 |

There are seven body executions and eight condition checks. The decreasing nonnegative counter gives us a direct termination argument: it approaches zero by one on each iteration.

In [2]:
remaining = 0
while remaining > 0:
    print("This line is never reached.")
print("A false initial condition produces zero iterations.")

A false initial condition produces zero iterations.


## 3. Choosing between `for` and `while`

The original countdown can be written more directly with `range`. Both versions below produce the same sequence.

In [3]:
for remaining in range(7, 0, -1):
    print(remaining, end=", ")
print("Done.")

7, 6, 5, 4, 3, 2, 1, Done.


A `for` loop can generally be expressed using a `while` loop and explicit iterator handling. That equivalence does not make `while` preferable in every situation. We should choose the construct that communicates the purpose most clearly.

| Task | Usually clearer choice | Reason |
|---|---|---|
| Visit each value in a collection | `for` | Iteration is managed automatically. |
| Repeat exactly ten times | `for` with `range` | The iteration count is explicit. |
| Request a value until it is valid | `while` | Completion depends on the input. |
| Refine an estimate until sufficiently accurate | `while` | Completion depends on the current state. |

For manually indexed loops, remember to update the index. For condition-controlled loops, identify a credible route to termination before running the code.

## 4. Accumulation: the largest admissible natural-number sum

Given a nonnegative integer limit `S`, find the largest integer `N` such that the sum of the first `N` positive integers does not exceed `S`. We include `S = 0`, for which `N = 0` and the empty sum is zero.

For example, when `S = 16`, the answer is `N = 5`: the sum is 15, whereas including 6 would produce 21.

The original algorithm adds terms while the current sum is below `S`, then returns `N - 1`. This overcorrects when the sum equals `S` exactly. A clearer solution checks whether the **next** term can be included before adding it.

In [4]:
def largest_admissible_n(limit):
    """Return (n, total) for the largest n with 1 + ... + n <= limit."""
    if type(limit) is not int or limit < 0:
        raise ValueError("limit must be a nonnegative integer")
    n = 0
    total = 0
    while total + (n + 1) <= limit:
        n += 1
        total += n
    return n, total

n, total = largest_admissible_n(16)
print(f"N = {n}; sum = {total}; next sum = {total + n + 1}")

N = 5; sum = 15; next sum = 21


At the start and end of each iteration, `total` equals the sum from 1 through `n`, and `total <= limit`. This is a **loop invariant**: a property maintained throughout the computation. When the loop stops, adding the next integer would exceed the limit. Together, these two facts establish that the result is admissible and maximal.

The loop also terminates: every accepted term is positive, the sum increases, and the limit is finite. Testing an exact triangular-number threshold is essential; testing only `S = 16` would miss the original error.

In [5]:
for limit, expected_n in [(0, 0), (1, 1), (2, 1), (3, 2),
                          (15, 5), (16, 5), (21, 6)]:
    n, total = largest_admissible_n(limit)
    assert n == expected_n
    assert total == n * (n + 1) // 2
    assert total <= limit < total + n + 1
    print(f"S = {limit:2d} -> N = {n}, sum = {total}")

S =  0 -> N = 0, sum = 0
S =  1 -> N = 1, sum = 1
S =  2 -> N = 1, sum = 1
S =  3 -> N = 2, sum = 3
S = 15 -> N = 5, sum = 15
S = 16 -> N = 5, sum = 15
S = 21 -> N = 6, sum = 21


## 5. Termination, iteration limits, and debugging

An iteration limit can prevent excessive computation, but reaching that limit does not prove that the original objective was achieved. We need to report why the loop stopped.

Consider this translated version of the source exercise. As pseudocode, its logic is valid: starting from 15 and subtracting 1 eventually makes `number > 4` false. If it is intended as Python, capitalization and colons must also be corrected (`while` and `print` are lowercase).

The large cap in the original exercise is not reached: the loop performs 11 iterations and ends at 4. Let us compare that outcome with a deliberately small cap.

In [6]:
def bounded_countdown(max_iterations):
    number = 15
    iterations = 0
    while number > 4 and iterations < max_iterations:
        number -= 1
        iterations += 1
    reason = "target reached" if number <= 4 else "iteration limit reached"
    return number, iterations, reason

for cap in (10**6, 5):
    print(f"Cap {cap}: {bounded_countdown(cap)}")

Cap 1000000: (4, 11, 'target reached')
Cap 5: (10, 5, 'iteration limit reached')


### Common causes of unintended nontermination

- The variable used in the condition is never updated.
- The update moves the state away from the stopping criterion.
- A `continue` statement skips an essential update.
- A floating-point value is compared for exact equality with a target it may never reach.
- External input never satisfies the required condition.

For example, the following fragment never changes `n`, so its condition remains true. It is shown as text to keep this notebook safe to run.

```python
n = 7
while n > 0:
    print(n)  # Missing: n -= 1
```

When debugging, inspect the state before and after one iteration. If you accidentally start an unbounded loop in Jupyter, interrupt the kernel execution before editing the cell.

## 6. `break`, `continue`, and post-test behavior

`break` exits the nearest enclosing loop immediately. `continue` skips the remainder of the current iteration and returns to the loop condition. Neither statement updates a counter automatically.

The following example prints odd integers from 1 to 7. The increment occurs **before** the possible `continue`, so even values cannot trap the loop.

In [7]:
number = 0
while number < 7:
    number += 1
    if number % 2 == 0:
        continue
    print(number, end=" ")
print()

1 3 5 7 


### Sentinel-controlled accumulation

A **sentinel** is a special value that signals the end of input. It must be distinguishable from valid observations. Here, `-1` terminates a stream of nonnegative measurements and is excluded from the sum. We use a finite iterator so that the example remains reproducible; exhaustion also provides a defined exit.

The body of `while True` runs before the stopping checks. This structure can therefore express post-test behavior, but its exit paths must be explicit.

In [8]:
measurements = iter([4, 7, 0, -1, 99])
total = 0
count = 0
while True:
    value = next(measurements, None)
    if value is None or value == -1:
        break
    total += value
    count += 1

mean = total / count if count else None
print(f"Count = {count}; total = {total}; mean = {mean}")
# 99 is not consumed: the sentinel has already ended processing.

Count = 3; total = 11; mean = 3.6666666666666665


Here, `None` denotes exhaustion and is not a permitted measurement. If `None` were valid data, we would need a different sentinel. The conditional expression for `mean` prevents division by zero when no observations are accepted.

`pass`, by contrast, performs no operation. It can serve as a syntactic placeholder, but it does not end an iteration or make a loop progress.

## 7. Understanding `while ... else`

An optional `else` suite executes when the loop condition becomes false, including when it is false on the very first check. It is skipped when the loop exits through `break`. A `return` from the enclosing function or an exception also bypasses the suite.

I find it useful to read this as “the loop ended through its condition,” rather than “the task succeeded.” Whether normal completion represents success depends on the algorithm.

In [9]:
for starting_value in (2, 0):
    remaining = starting_value
    while remaining > 0:
        remaining -= 1
    else:
        print(f"Start {starting_value}: condition is false; else executes.")

remaining = 2
while remaining > 0:
    break
else:
    print("This message is skipped because of break.")
print("The break example has finished.")

Start 2: condition is false; else executes.
Start 0: condition is false; else executes.
The break example has finished.


### A cap can also lead to the `else` suite

If the iteration limit is part of the condition, reaching it makes the condition false. The `else` suite then executes even if the underlying objective remains unmet.

In [10]:
error = 1.0
iterations = 0
while error > 0.01 and iterations < 2:
    error /= 2
    iterations += 1
else:
    print(f"Normal loop exit: error = {error}")
    print("Accuracy target met:", error <= 0.01)

Normal loop exit: error = 0.25
Accuracy target met: False


## 8. A reproducible guessing game

The original game chooses an integer from 1 to 33 and asks the learner to guess it. A positive guess receives a hint; a nonpositive guess ends the game. We retain these rules and add explicit outcomes for invalid text and exhausted sample input.

The function below receives a finite collection of guesses. Converting each item with `int` simulates the text received from `input`. Invalid text consumes a sample entry before `continue`, ensuring progress. The `while ... else` suite announces success only because this particular condition becomes false precisely when the correct number has been guessed.

In [11]:
def play_guessing_game(secret, guesses):
    """Demonstrate the game with a finite sequence of text guesses."""
    entries = iter(guesses)
    guess = None
    attempts = 0
    outcome = "sample input exhausted"
    while guess != secret:
        entry = next(entries, None)
        if entry is None:
            break
        attempts += 1
        try:
            guess = int(entry)
        except ValueError:
            print("Please enter an integer.")
            continue
        if guess <= 0:
            outcome = "cancelled"
            print("You have ended the game.")
            break
        if guess > secret:
            print("Your guess is too high.")
        elif guess < secret:
            print("Your guess is too low.")
    else:
        outcome = "correct guess"
        print("Congratulations—you found the number.")
    return outcome, attempts

print(play_guessing_game(17, ["ten", "10", "25", "17"]))

Please enter an integer.
Your guess is too low.
Your guess is too high.
Congratulations—you found the number.
('correct guess', 4)


In [12]:
print("Cancellation:", play_guessing_game(17, ["0"]))
print("Exhaustion:", play_guessing_game(17, ["12"]))

You have ended the game.
Cancellation: ('cancelled', 1)
Your guess is too low.
Exhaustion: ('sample input exhausted', 1)


To choose a random secret, use `random.randint(1, 33)`, which includes both endpoints. For reproducible experiments, a dedicated seeded generator is convenient. In a real game, do not print the secret.

In [13]:
import random

rng = random.Random(42)
secret = rng.randint(1, 33)
print("A secret in the interval 1–33 has been generated.")
assert 1 <= secret <= 33

A secret in the interval 1–33 has been generated.


For an interactive version, obtain a new entry with `input` inside the loop instead of advancing the sample iterator. Keep the cancellation rule and exception handling, and consider an attempt limit. The fragment below is intentionally commented out so that **Run All** does not wait for keyboard input.

In [14]:
# Optional interactive starting point:
# entry = input("Guess an integer from 1 to 33, or enter 0 to stop: ")
# Extend play_guessing_game by replacing sample retrieval with this prompt.
# If you add a maximum-attempt condition, test success separately after exit.

## 9. Validating an input domain

A domain constraint specifies which values a field permits. Validation can reject unsupported categories or malformed input, but it does not establish that an accepted value is factually correct. In data science, validation is therefore one component of data quality rather than a complete solution.

We use a study-mode field with two permitted values: `full-time` and `part-time`. The normalization step removes surrounding whitespace and standardizes letter case. In this schema, that normalization preserves the intended meaning.

In [15]:
allowed_modes = {"full-time", "part-time"}
sample_entries = iter(["weekends", " FULL-TIME "])
study_mode = ""
while study_mode not in allowed_modes:
    entry = next(sample_entries, None)
    if entry is None:
        print("No valid study mode was supplied.")
        break
    study_mode = entry.strip().casefold()
    if study_mode not in allowed_modes:
        print("Please choose full-time or part-time.")
else:
    print(f"Thank you. Recorded study mode: {study_mode}.")

Please choose full-time or part-time.
Thank you. Recorded study mode: full-time.


### Validate type conversion and range separately

Checking whether text can be converted to an integer is different from checking whether that integer is permitted. For instance, `"3.5"` is not an integer string, whereas `"-2"` is an integer string that fails a positive-value constraint.

In [16]:
entries = iter(["3.5", "-2", "7"])
requested_count = None
while True:
    entry = next(entries, None)
    if entry is None:
        print("Input exhausted before a valid count was obtained.")
        break
    try:
        candidate_count = int(entry)
    except ValueError:
        print("Please enter a whole number.")
        continue
    if candidate_count <= 0:
        print("The count must be positive.")
        continue
    requested_count = candidate_count
    break
print("Accepted count:", requested_count)

Please enter a whole number.
The count must be positive.
Accepted count: 7


## 10. Numerical iteration: approximation and convergence

Iterative numerical methods repeatedly refine an estimate. Their stopping criteria often combine an accuracy measure and a maximum iteration count. The original notes connect this idea with optimization in statistics and machine learning: stopping an algorithm does not, by itself, establish convergence or a globally optimal solution.

For a transparent example, we approximate the square root of 2 using Newton's update: replace `x` with `(x + 2 / x) / 2`. We begin at `x = 1`, use the residual `abs(x*x - 2)` as our accuracy measure, and impose a cap. This residual criterion concerns the squared value; it is not a claim of a specified number of correct decimal places in `x`.

The function is deliberately specific to this well-behaved example. A general numerical solver also needs to consider scaling, nonfinite values, invalid starting points, and problem-specific convergence properties.

In [17]:
def approximate_sqrt_two(tolerance=1e-10, max_iterations=20):
    """Return an estimate, residual, iteration count, and success flag."""
    if tolerance <= 0:
        raise ValueError("tolerance must be positive")
    if type(max_iterations) is not int or max_iterations < 0:
        raise ValueError("max_iterations must be a nonnegative integer")
    estimate = 1.0
    iterations = 0
    residual = abs(estimate * estimate - 2.0)
    while residual > tolerance and iterations < max_iterations:
        estimate = (estimate + 2.0 / estimate) / 2.0
        iterations += 1
        residual = abs(estimate * estimate - 2.0)
    return estimate, residual, iterations, residual <= tolerance

for cap in (20, 1):
    estimate, residual, iterations, converged = approximate_sqrt_two(
        max_iterations=cap
    )
    print(f"Cap {cap}: estimate={estimate:.12f}, residual={residual:.3e}, "
          f"iterations={iterations}, target met={converged}")

Cap 20: estimate=1.414213562375, residual=4.511e-12, iterations=4, target met=True
Cap 1: estimate=1.500000000000, residual=2.500e-01, iterations=1, target met=False


Notice that a cap of one produces an estimate, but the requested tolerance is not met. Reporting the estimate together with its residual and status makes the limitation visible.

Exact equality is usually a poor stopping criterion for floating-point approximations. Tolerances should reflect the scale and purpose of the calculation. Making a tolerance arbitrarily small can cause an algorithm to reach its cap because floating-point precision prevents further improvement.

## 11. Searching for the first `N` prime numbers

A prime number is an integer greater than 1 with no positive divisors other than 1 and itself. The original notebook first defines a primality test and then uses it within a `while` loop to print the first seven primes.

We retain that two-stage design. Our test checks candidate divisors only while their square is at most the candidate number. If a composite number has a factor larger than its square root, its paired factor is smaller than the square root; therefore, the smaller factor would already have been detected. This reduces unnecessary work compared with testing every integer below the candidate.

In [18]:
def is_prime(number):
    """Return whether an integer is prime."""
    if type(number) is not int:
        raise TypeError("number must be an integer")
    if number < 2:
        return False
    divisor = 2
    while divisor * divisor <= number:
        if number % divisor == 0:
            return False
        divisor += 1
    return True

for candidate in (1, 2, 9, 17, 49):
    print(f"{candidate}: {is_prime(candidate)}")

1: False
2: True
9: False
17: True
49: False


The outer search needs two separate quantities: the current candidate and the number of primes already found. Increment the candidate after **every** test, including when it is composite. Otherwise, the search could repeatedly test the same composite number.

The number of requested primes is known, but the number of candidate integers to examine is not specified in advance. This makes a condition-controlled loop a natural choice.

In [19]:
requested = 7
found = 0
candidate = 2
while found < requested:
    if is_prime(candidate):
        print(candidate, end=" ")
        found += 1
    candidate += 1
print()

2 3 5 7 11 13 17 


We can package the search as a function that returns a list for subsequent analysis. Here, a request for zero primes returns an empty list; negative or noninteger requests are rejected. This is a deliberate extension of the original positive-count exercise.

In [20]:
def first_primes(count):
    """Return the first count primes for a nonnegative integer count."""
    if type(count) is not int or count < 0:
        raise ValueError("count must be a nonnegative integer")
    primes = []
    candidate = 2
    while len(primes) < count:
        if is_prime(candidate):
            primes.append(candidate)
        candidate += 1
    return primes

print(first_primes(7))

[2, 3, 5, 7, 11, 13, 17]


There are infinitely many primes, so every finite nonnegative request has a mathematical answer. Nevertheless, this simple search can be slow for large requests. Termination and computational efficiency are different properties; an algorithm may eventually finish while still being unsuitable for a large workload.

## 12. Practice before consulting the solutions

### Exercise 1 — Trace selection and repetition

Translate the following pseudocode into Python and determine the output for `n = 8`, `n = 9`, and `n = 10`. The pseudocode's `for` endpoints are inclusive.

```text
INPUT n
IF n > 8
    WHILE n >= 4
        PRINT n
        n = n - 2
ELSE
    FOR i = 1 TO n
        PRINT i
```

Explain why 8 and 9 follow different branches and whether 4 is printed in each countdown.

### Exercise 2 — Interpret an iteration cap

Starting with `number = 15` and `iterations = 0`, repeatedly subtract 1 while `number > 4` and `iterations < max_iterations`. Predict the final number and iteration count for caps of 0, 5, and one million. Does normal loop termination always mean the target was reached?

### Exercise 3 — Generate primes

Write a function that returns the first `n` prime numbers. Explain the difference between the candidate value and the number of accepted primes. Check `n = 0`, `n = 1`, and `n = 7`.

### Exercise 4 — Repair a `continue` bug

Why does the fragment below become stuck? Rewrite it so that it prints the odd integers from 1 through 5.

```python
n = 0
while n < 5:
    if n % 2 == 0:
        continue
    print(n)
    n += 1
```

### Exercise 5 — Distinguish exit reasons

Modify a sentinel-controlled accumulator to return the total, the number of accepted observations, and an exit reason: `sentinel` or `exhausted`. Treat `-1` as the sentinel and all other supplied values as nonnegative numbers. Test an empty sequence and a sequence beginning with the sentinel.

In [21]:
# Your workspace: add your solutions below.
# Predict the output first, then compare it with the observed result.

## 13. Worked solutions and checks

### Exercise 1

The boundary `n = 8` follows the `else` branch and prints 1 through 8. The value 9 produces 9, 7, 5; the value 10 produces 10, 8, 6, 4. Subtracting 2 preserves parity, which explains why only the even countdown includes 4.

In [22]:
def trace_exercise(n):
    output = []
    if n > 8:
        while n >= 4:
            output.append(n)
            n -= 2
    else:
        for i in range(1, n + 1):
            output.append(i)
    return output

for n in (8, 9, 10):
    print(n, "->", trace_exercise(n))

8 -> [1, 2, 3, 4, 5, 6, 7, 8]
9 -> [9, 7, 5]
10 -> [10, 8, 6, 4]


### Exercise 2

A zero cap allows no iterations and leaves the number at 15. A cap of five leaves it at 10. A cap of one million is not exhausted: the target is reached after 11 iterations, leaving the number at 4. A cap limits work; it does not guarantee that the goal is achieved.

In [23]:
for cap in (0, 5, 10**6):
    print(cap, "->", bounded_countdown(cap))

0 -> (15, 0, 'iteration limit reached')
5 -> (10, 5, 'iteration limit reached')
1000000 -> (4, 11, 'target reached')


### Exercise 3

The `first_primes` function above is one solution. Its candidate advances on every iteration, whereas the result length increases only when a prime is found. These assertions check the requested boundary cases.

In [24]:
assert first_primes(0) == []
assert first_primes(1) == [2]
assert first_primes(7) == [2, 3, 5, 7, 11, 13, 17]
print("Prime-generation boundary checks passed.")

Prime-generation boundary checks passed.


### Exercise 4

In the faulty fragment, the first `continue` occurs at zero and skips the increment forever. Moving the increment before the parity test ensures progress on every iteration.

In [25]:
n = 0
while n < 5:
    n += 1
    if n % 2 == 0:
        continue
    print(n, end=" ")
print()

1 3 5 


### Exercise 5

The sentinel is excluded from both the total and the count. Exhaustion is a separate event, detected here with `None`, which is outside the permitted numeric input domain.

In [26]:
def accumulate_until_sentinel(values):
    entries = iter(values)
    total = 0
    count = 0
    while True:
        value = next(entries, None)
        if value is None:
            return total, count, "exhausted"
        if value == -1:
            return total, count, "sentinel"
        total += value
        count += 1

for values in ([], [-1], [4, 7, -1, 99], [4, 7]):
    print(values, "->", accumulate_until_sentinel(values))

[] -> (0, 0, 'exhausted')
[-1] -> (0, 0, 'sentinel')
[4, 7, -1, 99] -> (11, 2, 'sentinel')
[4, 7] -> (11, 2, 'exhausted')


### A compact verification suite

These checks focus on important failure modes: an exact cumulative-sum threshold, a composite square, an empty request, the distinction between convergence and an iteration cap, and whether a sentinel is excluded from accumulation. Passing examples is useful evidence, but the termination arguments remain an essential part of our reasoning.

In [27]:
assert largest_admissible_n(21) == (6, 21)
assert not is_prime(49)
assert trace_exercise(8) == list(range(1, 9))
assert trace_exercise(9) == [9, 7, 5]
assert trace_exercise(10) == [10, 8, 6, 4]
assert bounded_countdown(5) == (10, 5, "iteration limit reached")
assert approximate_sqrt_two()[3] is True
assert approximate_sqrt_two(max_iterations=1)[3] is False
assert accumulate_until_sentinel([]) == (0, 0, "exhausted")
assert accumulate_until_sentinel([-1]) == (0, 0, "sentinel")
assert accumulate_until_sentinel([4, 7, -1, 99]) == (11, 2, "sentinel")
try:
    first_primes(-1)
except ValueError:
    pass
else:
    raise AssertionError("Negative prime counts must be rejected.")
print("All verification checks passed.")

All verification checks passed.


## 14. Reflection and next steps

Before writing your next `while` loop, identify its initial state, continuation condition, and progress mechanism. Then ask what each possible exit means. A loop that stops has not necessarily solved its intended problem, and a result that looks plausible may still fail at a boundary.

I encourage you to revisit one example and write a brief explanation of why it terminates and why its result is correct. Being able to explain these two properties in ordinary language is as valuable as being able to write the code.

**Next topic:** simple data structures. Lists, tuples, sets, and dictionaries give us richer ways to organize the values that our loops process.

**Acknowledgment:** Adapted and expanded from Taufik Sutanto's 2019 teaching notebook. All stored outputs in this edition are generated from the revised examples.